# MDCE Colab + Google Drive Workflow

Use this notebook when the real dataset is in Google Drive.

Goal: load real/source data, map it into the MDCE schema, mark columns as real/derived/proxy, and run the same confidence engine used by the app.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone Or Pull Project Code

Replace the repo URL after you push the project to GitHub.

In [ ]:
%cd /content
# First time:
# !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git mdce

# Later runs:
# %cd /content/mdce
# !git pull


## 3. Install Dependencies

In [ ]:
# %cd /content/mdce
# !pip install -r requirements.txt

## 4. Point To Drive Dataset

Put downloaded/exported datasets here:

`/content/drive/MyDrive/MDCE/data/raw/`

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_DIR = Path('/content/mdce')
DRIVE_DIR = Path('/content/drive/MyDrive/MDCE')
RAW_DATA_PATH = DRIVE_DIR / 'data/raw/your_dataset.csv'  # change this
PROCESSED_DATA_PATH = DRIVE_DIR / 'data/processed/mdce_processed.csv'

sys.path.insert(0, str(PROJECT_DIR))


## 5. Inspect Raw Columns

Do this before renaming anything.

In [ ]:
raw = pd.read_csv(RAW_DATA_PATH)
print(raw.shape)
print(list(raw.columns))
raw.head()

## 6. Map Raw Columns To MDCE Schema

Edit this mapping based on the dataset you download.

Required minimum:

- `lap`
- `lap_time_s`

More real columns are better.

In [ ]:
# Example only. Replace keys/values based on actual raw columns.
column_map = {
    'LapNumber': 'lap',
    'LapTimeSeconds': 'lap_time_s',
    'Sector1Seconds': 'sector1_s',
    'Sector2Seconds': 'sector2_s',
    'Sector3Seconds': 'sector3_s',
    'Compound': 'tyre_compound',
    'TyreLife': 'tyre_age',
    'TrackStatus': 'track_status',
}

mapped = raw.rename(columns={old: new for old, new in column_map.items() if old in raw.columns})
mapped.head()

## 7. Load Through MDCE Validator

The loader will use real columns where present and derive/proxy only missing optional fields.

In [ ]:
from io import StringIO
from src.data_loader import load_race_csv

buffer = StringIO()
mapped.to_csv(buffer, index=False)
buffer.seek(0)

loaded = load_race_csv(buffer, source_name=str(RAW_DATA_PATH.name))
print('records:', len(loaded.records))
print('real:', loaded.real_columns)
print('derived:', loaded.derived_columns)
print('proxy:', loaded.proxy_columns)
print('\n'.join(loaded.warnings[:20]))

## 8. Run MDCE Analysis

In [ ]:
from src.models import ScenarioFlags
from src.pipeline import analyze_decision

result, scenario_records, notes, conflict = analyze_decision(
    loaded.records,
    ScenarioFlags(),
    prefer_granite=False,
)

print(result.recommendation)
print(result.confidence)
print('conflict:', conflict)
print(result.explanation)

## 9. Export Processed CSV For App Upload

In [ ]:
PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame([record.__dict__ for record in loaded.records]).drop(columns=['missing']).to_csv(PROCESSED_DATA_PATH, index=False)
print(PROCESSED_DATA_PATH)

## 10. Provenance Note For README

Before final submission, write:

- source link
- licence
- raw file name
- real columns used
- derived columns
- proxy columns
- processing steps